# Detección de incendios con redes neuronales convolucionales

**Proyecto Final de Ingeniería en Electrónica — FaCENA-UNNE**
(Red de sensores LoRa para detección temprana de incendios)

Entrenamos una **CNN** en TensorFlow/Keras que da, para cada imagen, tres
probabilidades excluyentes (**softmax**, suman 1):

| Salida | Qué mide |
|---|---|
| P(`normal`) | no hay ni fuego ni humo |
| P(`humo`) | hay humo pero no llamas visibles |
| P(`incendio`) | hay llamas visibles, haya o no humo |

¿Por qué 3 clases y no separar fuego de fuego+humo? De día casi todo incendio tiene
humo: el 80 % de las imágenes con fuego también lo tiene, así que separar la clase
escasa en `fuego` (5,4 %) y `fuego+humo` (21,6 %) la partía en dos. Juntas, `incendio`
es el 27 % del banco y la red aprende la llama haya humo o no.

La entrada es **QVGA 320×240 RGB** (el color es la señal del fuego) y el dataset es
**D-Fire** (`sayedgamal99/smoke-fire-detection-yolo` de Kaggle, 21527 imágenes JPEG),
un banco de *detección* (bounding boxes YOLO: `0` = humo, `1` = fuego) del que derivamos
la categoría a nivel imagen (ver §2).

> **Base de este notebook.** Es una adaptación del notebook de clasificación de tumores
> cerebrales en MRI: mismo pipeline `tf.data`, mismo régimen de entrenamiento
> (AdamW + EMA, EarlyStopping, ReduceLROnPlateau) y misma estructura de evaluación
> (matriz de confusión, Grad-CAM). Cambia lo que depende del problema: dataset D-Fire en
> lugar de BRISC2025, imágenes **RGB a color** en vez de escala de grises, categorías
> derivadas de anotaciones YOLO y **3 clases excluyentes** (softmax) de salida en lugar
> de 4.
>
> **La arquitectura ya NO es la CNN de cuatro bloques** de aquel notebook: el modelo
> tiene que correr en el nodo ESP32-S3 y aquella red (1,1 M de parámetros, 845 MMAC)
> tardaba ~8 s por inferencia, así que no era desplegable. Se usa **MobileNetV1 α=0.25**
> preentrenada en ImageNet (§4); la cuantización y la exportación a **TFLite Micro
> int8** están en `5_cuantizacion.ipynb`.

**Recorrido:** exploración de datos → pipeline `tf.data` → arquitectura y costo en el
nodo → entrenamiento → evaluación float → Grad-CAM → modelo entrenado (`.keras`). La
cuantización a int8, la evaluación del modelo cuantizado "como lo ve la cámara", el
umbral de decisión y los archivos para el firmware siguen en `5_cuantizacion.ipynb`.

Link del dataset: https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo

## 0. Descarga del dataset

Descargamos **D-Fire** en formato YOLO (bounding boxes para humo y fuego) desde
Kaggle: [`sayedgamal99/smoke-fire-detection-yolo`](https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo).
Con `KAGGLEHUB_CACHE` apuntando a `data/` del proyecto, el dataset queda dentro del
proyecto (no en `~/.cache/kagglehub`), y la celda es **idempotente**: la segunda
corrida no vuelve a descargar.
Requiere credencial de Kaggle en `~/.kaggle/kaggle.json` o las variables de
entorno `KAGGLE_USERNAME` / `KAGGLE_KEY`.

In [ ]:
from pathlib import Path
import os

# El kernel puede arrancar en la raíz del proyecto o en `notebooks/`: resolvemos
# `data/` de forma robusta en ambos casos.
RAIZ = Path("data") if Path("data").exists() else Path("../data")
RAIZ = RAIZ.resolve()

# `kagglehub` descarga donde diga KAGGLEHUB_CACHE: hay que setearlo ANTES de
# importarlo para que el dataset quede en `data/` del proyecto y no en
# ~/.cache/kagglehub.
os.environ["KAGGLEHUB_CACHE"] = str(RAIZ)
import kagglehub

# Si el dataset ya está descargado, devuelve la ruta y no vuelve a bajarlo.
DATASET_DIR = Path(kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo"))
print("Dataset en:", DATASET_DIR)
for ruta in sorted(DATASET_DIR.iterdir()):
    print(" ", ruta.name)
    if ruta.is_dir():
        print("   ", sorted(s.name for s in ruta.iterdir()))

## 1. Configuración del entorno

Antes de importar TensorFlow precargamos las librerías CUDA. Los *wheels* de TF 2.21
no registran correctamente las rutas de los paquetes `nvidia-*` instalados por pip, así
que las cargamos a mano; sin esto TF cae a CPU silenciosamente.

Y un detalle que cuesta una noche de trabajo si no se sabe: TF 2.21 arrastra
**cuDNN 9.26, que ya no trae kernels para Pascal**. En una GTX 1050 (sm_6.1) la
primera convolución muere con `No algorithm worked! ... cudnn status: 5003` — no es
falta de memoria, es que no existe el kernel. `pyproject.toml` fija
`nvidia-cudnn-cu12==9.8.0.87`, la última que sí compila para esa arquitectura;
verificado en esta máquina. La sonda de abajo es la red de seguridad: si igual no hay
convolución posible en la GPU, esconde el dispositivo y se entrena en CPU en vez de
reventar veinte celdas más adelante.

In [ ]:
# Precarga de librerías CUDA (debe ejecutarse ANTES de importar tensorflow)
import ctypes
import glob
import os
import site
import subprocess
import sys

for _lib in glob.glob(os.path.join(site.getsitepackages()[0], "nvidia", "*", "lib", "*.so*")):
    try:
        ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass

# Sonda de GPU en un subproceso: cuDNN 9.26 (el que trae TF 2.21) ya no tiene
# kernels para Pascal (sm_6.1, la GTX 1050 de esta máquina), así que la primera
# convolución muere con "No algorithm worked!". Averiguarlo acá y caer a CPU es
# más barato que descubrirlo recién en `fit`. Va en un subproceso porque
# `set_visible_devices` no se puede llamar después de inicializar el contexto.
_SONDA = (
    "import ctypes,glob,os,site\n"
    "[ctypes.CDLL(l, mode=ctypes.RTLD_GLOBAL) for l in "
    "glob.glob(os.path.join(site.getsitepackages()[0],'nvidia','*','lib','*.so*'))"
    " if os.path.exists(l)]\n"
    "import tensorflow as tf\n"
    "assert tf.config.list_physical_devices('GPU')\n"
    "with tf.device('/GPU:0'):\n"
    "    tf.nn.conv2d(tf.zeros([1,32,32,3]), tf.zeros([3,3,3,8]), 1, 'SAME').numpy()\n"
)
_sonda = subprocess.run([sys.executable, "-c", _SONDA], capture_output=True,
                        env={**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3"})
GPU_USABLE = _sonda.returncode == 0
if not GPU_USABLE:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("GPU descartada, se entrena en CPU. Motivo:")
    print("  " + (_sonda.stderr.decode().strip().splitlines() or ["sin GPU"])[-1][:160])
else:
    print("GPU utilizable para convoluciones.")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42                             # Semilla para reproducibilidad
keras.utils.set_random_seed(SEED)

# `memory_growth` evita que TF reserve toda la VRAM de entrada: útil en GPUs chicas.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Dispositivos:", tf.config.list_physical_devices())

## 2. Exploración del dataset

D-Fire es un banco de *detección*: cada imagen trae un `.txt` con cajas YOLO
(`0` = humo, `1` = fuego; archivo vacío = fondo, sin objetos). Para clasificar a nivel
imagen derivamos la **categoría** de cada archivo desde sus cajas: `incendio` si hay al
menos una caja de fuego (`1`), haya o no humo; si no, `humo` si hay al menos una caja de
humo (`0`); si no, `normal`.

La historia en dos frases: primero este notebook usó tres clases excluyentes, después
pasó a dos etiquetas independientes [fuego, humo] con sus cuatro combinaciones, y ahora
vuelve a 3 clases pero con `incendio` = llama con o sin humo — porque separar `fuego`
de `fuego+humo` partía en dos la clase más escasa y no le servía a la alerta. La lógica
está en `modulos/dfire.py`, compartida con el notebook de cuantización.

Recorremos los tres splits que trae el banco (`train`/`val`/`test`) y **conservamos a
cuál pertenece cada imagen**: se usa esa partición tal cual, no una nueva (§3). Armamos
de una vez el índice de rutas, etiquetas y split: lo vamos a reutilizar para el pipeline
y para la evaluación.

In [ ]:
# Etiquetas, partición y lectura de imágenes viven en `modulos/dfire.py`: el notebook de
# cuantización (5) tiene que preprocesar EXACTAMENTE igual que acá, y una sola
# definición importada por los dos lo garantiza.
import sys

MODULOS = Path("../modulos") if Path("../modulos").exists() else Path("modulos")
if not MODULOS.exists():
    # Kernel remoto (extensión Colab de VS Code): los archivos locales no viajan al
    # runtime, así que se trae `modulos/` del repo público.
    import subprocess
    if Path("repo").exists():       # ya clonado: traer lo último, no usar una copia vieja
        subprocess.run(["git", "-C", "repo", "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/fgmolteni/proyecto-final-pruebas", "repo"], check=True)
    MODULOS = Path("repo/modulos")
SALIDAS = MODULOS.parent / "salidas"        # misma raíz que `modulos/`
sys.path.insert(0, str(MODULOS.resolve()))
# Tras un `git pull` el kernel conserva la versión vieja ya importada: sin esto,
# re-correr la celda sigue viendo el dfire / nn_esp32 anterior hasta reiniciar.
for _m in ("dfire", "nn_esp32"):
    sys.modules.pop(_m, None)
from dfire import CATEGORIAS, COMBOS, categoria, combo, indexar

DATA_DIR = DATASET_DIR / "data"     # local: data/datasets/...; Colab: caché de kagglehub

# `indexar` conserva de qué split oficial vino cada imagen: D-Fire son frames de video
# y rearmar la partición al azar dejaría cuadros casi idénticos de un mismo incendio
# en train y en test. La partición que trae el banco ya evita eso.
ARCHIVOS, ETIQUETAS, SPLITS = indexar(DATA_DIR)     # ETIQUETAS: (N, 2) [fuego, humo]
CAT = categoria(ETIQUETAS)                          # (N,): 0 normal, 1 humo, 2 incendio
conteos = np.bincount(CAT, minlength=len(CATEGORIAS))

for i, nombre in enumerate(CATEGORIAS):
    print(f"{nombre:<10} {conteos[i]:>6}  ({100 * conteos[i] / len(CAT):5.1f} %)")
print(f"{'TOTAL':<10} {len(CAT):>6}")

# `incendio` junta `fuego` (COMBOS[2]) y `fuego+humo` (COMBOS[3]): la fusión suma las dos.
con_humo = np.sum(combo(ETIQUETAS[CAT == 2]) == 3)
print(f"\nincendio: {con_humo} con humo y {np.sum(CAT == 2) - con_humo} sin humo "
      f"({100 * con_humo / np.sum(CAT == 2):.0f} % con humo).")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(CATEGORIAS, conteos, color="blue")
ax.set_ylabel("Cantidad de imágenes")
ax.set_title("Categorías")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

Miramos una muestra de cada clase. Son fotos a color de escenas variadas —interiores,
rutas, campos, de día y de noche— en tamaños dispares, por lo que habrá que llevarlas
todas a una resolución común. Esa variedad de encuadres e iluminación es lo que el
aumento de datos tiene que absorber para que la red no memorice fondos.

In [ ]:
fig, axes = plt.subplots(len(CATEGORIAS), 4, figsize=(9, 7))
for fila, nombre in enumerate(CATEGORIAS):
    archivos = ARCHIVOS[CAT == fila][:4]
    for col, archivo in enumerate(archivos):
        img = keras.utils.load_img(archivo)
        axes[fila, col].imshow(img)
        axes[fila, col].axis("off")
        titulo = f"{nombre}  —  " if col == 0 else ""
        axes[fila, col].set_title(f"{titulo}{img.size[0]}×{img.size[1]}", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Pipeline de datos

Decisiones de preprocesamiento:

- **Tamaño 320×240 (QVGA).** Es un `framesize` nativo del OV2640 (`FRAMESIZE_QVGA`):
  el sensor entrega el buffer ya del tamaño que come la red y el firmware **no necesita
  código de reescalado**. Eso elimina la clase de error más cara del despliegue: que el
  resize de la placa no coincida con el del entrenamiento (bilineal vs *nearest*,
  recorte vs estirado) y la exactitud se caiga sin que nada falle. Además QVGA usa el
  campo 4:3 completo del sensor, mientras que los tamaños cuadrados (96×96, 128×128,
  240×240, 320×320) recortan una ventana central de 1200×1200 y pierden el 25 % del campo
  horizontal. A 96×96 el recall de fuego se caía al ~60 % con cajas de menos de 5 px de
  lado (vs ~93 % con cajas más grandes); a QVGA casi no quedan cajas tan chicas (ver
  celda siguiente). D-Fire acompaña: el 97 % de sus imágenes tiene lado menor ≥ 320 px
  (mediana 687 px), así que QVGA no sobremuestrea, y estirar una foto 16:9 (la mediana
  del banco) a 4:3 la comprime 1,33× en horizontal (a cuadrado era 1,78×). El precio está
  en el entrenamiento, no en el nodo: el caché en RAM sube a ~5 GB (0,6 GB a 96×96).
- **RGB (3 canales).** El color es LA señal discriminativa: el naranja de la llama y el
  gris del humo no existen en escala de grises. Se conserva la imagen a color tal cual.
- **Partición oficial del banco** (14 122 / 3 099 / 4 306), no una nueva. D-Fire se armó
  en buena medida con cuadros extraídos de video: hay imágenes casi idénticas entre sí.
  Repartir al azar el banco completo manda cuadros hermanos del mismo incendio a
  entrenamiento y a prueba, y el número de prueba deja de medir generalización para medir
  memorización. La partición que trae el banco ya está hecha con ese cuidado, así que
  respetarla es a la vez lo correcto y lo más barato. Sólo desordenamos *dentro* de cada
  split, para que ningún lote quede con una sola clase.
- **Cuánto sobra de todos modos**: ni siquiera la partición oficial queda limpia. Medido
  en §6bis, el **22,9 %** de las imágenes de prueba tiene un gemelo perceptual en
  entrenamiento. Por eso la métrica que vale se informa dos veces: sobre la prueba
  completa y sobre el subconjunto sin gemelos.

In [ ]:
from dfire import particion

IMG_SIZE = (240, 320)   # (alto, ancho): QVGA del OV2640. (96, 96) reproduce la línea base.
BATCH_SIZE = 32

# Partición oficial del banco. `particion` sólo desordena DENTRO de cada split: la
# pertenencia no cambia, pero ningún lote queda con una sola clase.
particiones = particion(SPLITS, SEED)
idx_train, idx_val, idx_test = (particiones[s] for s in ("train", "val", "test"))

for nombre, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    reparto = "  ".join(f"{CATEGORIAS[c]} {np.sum(CAT[idx] == c)}"
                        for c in range(len(CATEGORIAS)))
    print(f"{nombre:<6} {len(idx):>6} imágenes    {reparto}")

### ¿Cuántos píxeles tiene un incendio chico?

Antes de fijar la resolución medimos, para cada `framesize` candidato del OV2640, en
cuántos píxeles cae la caja de fuego más grande de cada imagen con fuego (lado
equivalente `sqrt(w·h)` llevado a cada tamaño). Debajo de ~5 px de lado no queda llama
visible que aprender: esos fallos son ruido de etiqueta, no de la red.

In [ ]:
from dfire import cajas_imagen

CANDIDATOS = [("96×96", (96, 96)), ("128×128", (128, 128)), ("QCIF", (144, 176)),
              ("HQVGA", (176, 240)), ("240×240", (240, 240)), ("QVGA", (240, 320)),
              ("CIF", (296, 400)), ("VGA", (480, 640))]

# Lado relativo (fracción del lado de la imagen) de la caja de fuego más grande, sobre
# las imágenes con fuego de TODO el banco: el mismo número sirve para todos los
# tamaños candidatos, sólo cambia la escala `sqrt(alto·ancho)`.
lados_rel = []
for ruta in ARCHIVOS[ETIQUETAS[:, 0] == 1]:
    txt = Path(ruta).parent.parent / "labels" / (Path(ruta).stem + ".txt")
    cajas = np.asarray(cajas_imagen(txt), dtype=float).reshape(-1, 5)
    fuego = cajas[cajas[:, 0] == 0]     # canal 0 = fuego
    lados_rel.append(np.sqrt(float(np.max((fuego[:, 3] - fuego[:, 1])
                                          * (fuego[:, 4] - fuego[:, 2])))) if len(fuego) else 0.0)
lados_rel = np.array(lados_rel)

print(f"{len(lados_rel)} imágenes con fuego en el banco\n")
print(f"{'framesize':<10}{'tamaño':<12}{'< 5 px':>8}{'< 8 px':>8}{'mediana':>10}")
for nombre, (alto, ancho) in CANDIDATOS:
    px = lados_rel * np.sqrt(alto * ancho)
    print(f"{nombre:<10}{alto}×{ancho:<8}{np.mean(px < 5):>8.1%}{np.mean(px < 8):>8.1%}"
          f"{np.median(px):>9.1f} px")
# Referencia: a 96×96 el 14,7 % queda debajo de 5 px; a QVGA, el 0,8 %.
# Por eso QVGA: casi no quedan incendios sin llama visible que aprender.

In [ ]:
from dfire import armar_ds

# `armar_ds` lee cada imagen con `dfire.leer_imagen` (bilineal con antialias, redondeo a
# uint8), la cachea en RAM y la entrega en lotes float32 0-255: la escala a [-1, 1] está
# adentro del modelo (`Rescaling`, ver §4). El notebook de cuantización arma sus datos
# con la misma función, así que calibración y evaluación int8 ven exactamente lo mismo.
# A QVGA el caché sube a ~5 GB (0,6 GB a 96×96): es el precio del entrenamiento, no del nodo.
train_ds = armar_ds(ARCHIVOS[idx_train], CAT[idx_train], BATCH_SIZE,
                    mezclar=True, seed=SEED, tam=IMG_SIZE)
val_ds = armar_ds(ARCHIVOS[idx_val], CAT[idx_val], BATCH_SIZE, tam=IMG_SIZE)
test_ds = armar_ds(ARCHIVOS[idx_test], CAT[idx_test], BATCH_SIZE, tam=IMG_SIZE)

# Un lote de entrenamiento tiene que ver las tres categorías.
_, y0 = next(iter(train_ds))
print(f"categorías distintas en el primer lote: {len(np.unique(y0.numpy()))} "
      f"de {len(CATEGORIAS)}")

### Escala de la entrada

Los píxeles llegan en 0–255. El modelo arranca con `layers.Rescaling(1/127.5, offset=-1)`:
una escala **fija**, sin estadísticas que medir, que deja la entrada en [-1, 1] — que es
exactamente lo que espera MobileNet preentrenada en ImageNet (`preprocess_input` hace eso).

Fija y no adaptada (`layers.Normalization`) por la cuantización. Con una escala conocida,
el converter pliega el `Rescaling` dentro del grafo y deja la entrada del `.tflite` en
`scale = 1`, `zero_point = -128`, o sea que el preprocesamiento en el firmware es una
resta de una constante:

```c
input[i] = pixel[i] - 128;   // píxel 0-255 del OV2640 (uint8) -> int8
```

Con media y varianza medidas, esas constantes salen del dataset de entrenamiento y hay
que replicarlas a mano en el nodo.


### Aumento de datos

Con ~15 mil imágenes de entrenamiento el sobreajuste es el riesgo principal, y hay uno
específico que evitar: como el banco mezcla encuadres, fondos e iluminaciones, la red
podría aprender *dónde* suele aparecer el fuego en el cuadro (o qué fondo lo acompaña)
en lugar de *cómo* se ve. Espejado, rotación, traslación, zoom y contraste generan en
cada época una variante distinta de la misma imagen, así que esas pistas dejan de ser
confiables. Sumamos **brillo aleatorio** (`RandomBrightness`): las escenas de fuego
varían muchísimo en iluminación (día/noche, exposición), y la red no debe depender del
nivel absoluto de luz.

Al final agregamos **borrado aleatorio** (`RandomErasing`): en cada época se tapa un
parche al azar. Como el parche cae en cualquier lado, ninguna región concreta está
siempre disponible, y la red no puede resolver la clase mirando un solo lugar fijo del
cuadro. Es la misma idea que `Dropout`, aplicada a la entrada en vez de a las
activaciones.

Sin relleno negro: las fotos no tienen fondo uniforme, así que rotación, traslación y
zoom usan el modo `reflect` por defecto (las esquinas vacías se espejan en vez de
rellenarse con un color que no existe en la escena).

Las capas de aumento sólo actúan en entrenamiento: en inferencia dejan pasar la imagen
sin tocar.

In [ ]:
aumento_datos = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(1 / 12),      # factor en fracciones de 2*pi -> ±30°
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.10),
        # Las escenas de fuego varían mucho en iluminación (día/noche, exposición):
        # el brillo aleatorio evita que la red dependa del nivel absoluto de luz.
        layers.RandomBrightness(0.15),
        # Borrado aleatorio: tapa un parche de la imagen para que la red no pueda
        # apoyarse en una sola región. factor=0.5 -> probabilidad por imagen sorteada
        # en [0, 0.5]; scale es la fracción de área borrada, acotada al 15 % para no
        # tapar el foco de fuego entero y ensuciar la etiqueta.
        layers.RandomErasing(factor=0.5, scale=(0.02, 0.15)),
    ],
    name="aumento_datos",
)

In [ ]:
# Efecto del aumento sobre una misma imagen
lote_img, _ = next(iter(train_ds))
original = lote_img[0]

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(original.numpy().astype("uint8"))
axes[0].set_title("original")
axes[0].axis("off")
for i in range(1, 6):
    aumentada = aumento_datos(tf.expand_dims(original, 0), training=True)
    axes[i].imshow(aumentada[0].numpy().astype("uint8"))
    axes[i].set_title(f"variante {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 4. Arquitectura de la red

**MobileNetV1 con α=0.25**, preentrenada en ImageNet, más una cabeza de clasificación:

```
240×320×3  →  Rescaling(1/127.5, -1)         escala fija a [-1, 1]
           →  MobileNetV1 α=0.25 sin top     14 bloques, salida 8×10×256
           →  GlobalAveragePooling2D   →  256
           →  Dropout(0.4)  →  Dense(3, softmax)       P(normal), P(humo), P(incendio)
```

Se entrenan **dos modelos que comparten pesos**: `modelo_train` (`aumento_datos` →
`modelo_base`) es el que se compila y entrena; `modelo_base` (píxeles 0–255 → escala →
backbone → cabeza) es el que se evalúa y se exporta. El aumento no puede quedar dentro del
modelo exportado: `RandomRotation`, `RandomTranslation` y `RandomZoom` emiten
`ImageProjectiveTransformV3`, que no es una op *builtin* de TFLite y TFLite Micro no tiene
*flex delegate* para resolverla.

### Por qué MobileNet y no la CNN de cuatro bloques

El requisito no es "clasificar bien" sino "clasificar en un ESP32-S3". La CNN de cuatro
bloques tenía 1,1 M de parámetros y 845 MMAC a 128×128: bajar la resolución no la salvaba,
el problema era la red. Números calculados con `modulos/nn_esp32.py` (anclado en la
latencia medida de `person_detection` con ESP-NN):

| Variante | MMAC | Pesos int8 | Arena est. | Dónde | Latencia S3 |
|---|---|---|---|---|---|
| CNN de 4 bloques 128×128 | 845 | 1146 kB | 1331 kB | PSRAM | ~8 s |
| CNN de 4 bloques 96×96 | 475 | 1146 kB | 749 kB | PSRAM | ~4,5 s |
| MobileNetV1 α=0.25 96×96×3 | 7,5 | 206 kB | 70 kB | SRAM | ~57 ms |
| **MobileNetV1 α=0.25 QVGA 320×240** | **62,9** | **206 kB** | **~585 kB** | **PSRAM** | **~0,6 s** |
| MobileNetV1 α=0.5 QVGA 320×240 | 230 | 796 kB | ~1,2 MB | PSRAM | ~2,2 s |

Son estimaciones del modelo analítico, no mediciones en placa. La columna de pesos es el
conteo de parámetros: el `.tflite` exportado pesa un poco más porque cada convolución
guarda además su escala y `zero_point` por canal. El tiempo de inferencia NO importa en
el nodo (una foto cada varios minutos): lo que limita es entrenar (RAM, GPU de 4 GB,
~8× más tiempo por época) y la resolución del propio banco. Además del tamaño, dos
razones técnicas:

- **Convoluciones *depthwise separable*.** Factorizan una convolución 3×3 en una espacial
  por canal (*depthwise*) más una 1×1 que mezcla canales (*pointwise*): ~8× menos MACs para
  el mismo campo receptivo.
- **`ReLU6` en vez de `ReLU`.** Acota las activaciones a [0, 6] en lugar de dejarlas
  abiertas. En int8 el rango de cada tensor se mapea a 256 niveles: un techo acotado evita
  que un *outlier* se lleve todo el rango y aplaste la resolución del resto. Es la razón
  por la que las redes para microcontroladores usan ReLU6.

### Elementos de la cabeza

- **`GlobalAveragePooling2D` en lugar de `Flatten`.** Un `Flatten` acá produciría un vector
  de 8·10·256 = 20480 elementos y una capa densa con ~60 mil parámetros de más; el promedio
  global colapsa cada mapa a un escalar: 256 valores y cero parámetros.
- **`Dropout(0.4)`** antes de la salida. El backbone ya viene regularizado por ImageNet.
- **`Softmax` de 3 salidas.** Las clases son excluyentes y suman 1: cada imagen es normal,
  humo o incendio, sin salidas a medio encender.
- **`Rescaling` dentro del modelo.** Queda autocontenido: recibe píxeles crudos 0–255 y no
  depende de ningún preprocesamiento externo al exportarlo.

### Preentrenado con entrada no cuadrada

`weights="imagenet"` sólo publica filas 128/160/192/224: con 240×320 Keras avisa y carga
los pesos de 224 igual. Funciona —las convoluciones no dependen del tamaño de
entrada— y sólo hay que ignorar el warning. El preentrenado es lo que hace viable
entrenar con 15 mil imágenes una red de 219 k parámetros.

### Optimizador

Dos cambios sobre el `Adam` de manual, ninguno toca la arquitectura:

- **`AdamW` en lugar de `Adam`.** El decaimiento de pesos se aplica como tal, restando
  una fracción del peso en cada paso. `Adam` lo implementa sumando el término al
  gradiente, donde el momento y el escalado adaptativo lo deforman: los pesos con
  gradiente chico terminan decayendo mucho más que los de gradiente grande, que es lo
  contrario de lo que se busca. El `weight_decay` queda en el valor por defecto (0,004).
- **Promedio móvil de los pesos (`use_ema`).** En vez de quedarse con el punto exacto
  donde cortó el entrenamiento, se usa el promedio de los últimos pasos: el mínimo de la
  pérdida es una región ruidosa y el promedio cae más cerca del centro que cualquier
  iteración suelta.

Hay una trampa en el segundo, y por eso lleva `ema_overwrite_frequency`: Keras vuelca el
promedio en los pesos al terminar `fit()`, pero eso pasa **antes** de que
`EarlyStopping(restore_best_weights=True)` restaure los pesos de la mejor época. Sin ese
argumento, el promedio se calcula, se escribe y se pisa — los pesos finales salen
idénticos a no haber activado la opción. Fijándolo en los pasos de una época, el
promedio se escribe cada época, la validación mide el modelo promediado y EarlyStopping
selecciona sobre él.

In [ ]:
ALPHA = 0.25      # factor de ancho de MobileNetV1: 219 k parámetros, ~206 kB en int8


def construir_modelo_base(n_clases):
    '''Píxeles 0-255 -> escala fija -> MobileNetV1 -> cabeza de clasificación.

    Es el modelo EXPORTABLE: no tiene capas de aumento, así que en el grafo no aparece
    ninguna `ImageProjectiveTransformV3` (op que TFLite Micro no sabe ejecutar).
    '''
    entrada = keras.Input(shape=(*IMG_SIZE, 3), name="entrada")
    # Escala fija a [-1, 1]: es la normalización que espera MobileNet preentrenada
    # (`preprocess_input`). El converter la pliega en el grafo: la entrada del .tflite
    # queda int8 con scale=1, zp=-128 -> en el firmware es `pixel - 128` (ver §3).
    x = layers.Rescaling(1 / 127.5, offset=-1, name="rescaling")(entrada)

    backbone = keras.applications.MobileNet(
        alpha=ALPHA, include_top=False, weights="imagenet",
        input_shape=(*IMG_SIZE, 3), name="mobilenet",
    )

    # Etapa 1: backbone congelado. La cabeza arranca al azar y sus gradientes
    # iniciales son enormes; descongelar desde el primer lote borra las
    # características de ImageNet. Se descongela en §5 con lr/10.
    backbone.trainable = False

    x = backbone(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    # Softmax: clases excluyentes, las tres probabilidades suman 1.
    salida = layers.Dense(n_clases, activation="softmax", name="salida")(x)
    return keras.Model(entrada, salida, name="cnn_fuego_base")


modelo_base = construir_modelo_base(len(CATEGORIAS))

# El aumento sólo existe en entrenamiento: se lo pone por delante del modelo base, no
# adentro. Los dos modelos comparten los pesos (son las mismas capas).
entrada_train = keras.Input(shape=(*IMG_SIZE, 3), name="entrada_train")
modelo_train = keras.Model(
    entrada_train, modelo_base(aumento_datos(entrada_train)), name="cnn_fuego_train"
)

# sparse_categorical_crossentropy: la etiqueta es el índice entero de la categoría
# (armar_ds la entrega float32 y Keras la castea). La métrica compara el argmax con
# ese índice, que es exactamente la regla de decisión.
#
# ema_overwrite_frequency = pasos de una época. Sin este argumento el promedio móvil
# se escribe en los pesos recién al terminar `fit`, y como eso ocurre ANTES de que
# EarlyStopping restaure los mejores pesos, el promedio se pierde: los pesos finales
# quedan idénticos a no haber usado EMA. Escribiéndolo cada época, la validación mide
# el modelo promediado y EarlyStopping guarda y restaura ese.
modelo_train.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-3,
        use_ema=True,                       #ema: Exponential Moving Average
        ema_momentum=0.99,
        ema_overwrite_frequency=len(train_ds),
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
modelo_train.summary()

In [ ]:
# ¿Entra esta variante en el nodo? Cuesta segundos y evita entrenar una noche entera
# para descubrir que no. Es el mismo módulo con el que se eligió el hardware.
# `modulos/` ya está en el path (§2).
import nn_esp32

perfil_esp32 = nn_esp32.perfil(alpha=ALPHA, res=IMG_SIZE, canales_in=3,
                               clases=len(CATEGORIAS))
donde, arena_kb = nn_esp32.donde_entra(perfil_esp32)
latencia_s3 = nn_esp32.latencia_ms(perfil_esp32, "esp32s3", en_psram=(donde == "PSRAM"))

print(f"MobileNetV1 α={ALPHA}  {IMG_SIZE[1]}×{IMG_SIZE[0]}×3  {len(CATEGORIAS)} salidas softmax")
print(f"  MACs        {perfil_esp32['macs'] / 1e6:8.2f} MMAC")
print(f"  pesos int8  {perfil_esp32['flash_kb']:8.0f} kB  (flash, array const)")
print(f"  arena pico  {arena_kb:8.0f} kB  (incluye 30 % de scratch)")
print(f"  latencia S3 {latencia_s3:8.0f} ms  (extrapolada de person_detection + ESP-NN)")
print(f"  veredicto   {donde}")
print(f"  energía por captura (cámara + filtro clásico + red): "
      f"{nn_esp32.energia_captura(perfil_esp32, 'esp32s3')['total']:.0f} mJ")

# Costo por framesize candidato (α 0.25 y 0.5): con qué se paga cada resolución.
# El tiempo no importa en el nodo (una foto cada varios minutos); lo que limita es
# entrenar (caché en RAM, GPU de 4 GB) y la resolución del propio banco.
print(f"\n{'framesize':<10}{'α':<6}{'MMAC':>8}{'pesos kB':>9}{'arena kB':>9}"
      f"{'dónde':>7}{'lat S3 ms':>10}{'caché GB':>9}")
for nombre, (alto, ancho) in CANDIDATOS:
    for a in (0.25, 0.5):
        p = nn_esp32.perfil(alpha=a, res=(alto, ancho), canales_in=3,
                            clases=len(CATEGORIAS))
        # `lugar`, no `donde`: `donde` es el del modelo entrenado y va al barrido (§8).
        lugar, arena = nn_esp32.donde_entra(p)
        lat = nn_esp32.latencia_ms(p, "esp32s3", en_psram=(lugar == "PSRAM"))
        cache_gb = len(ARCHIVOS) * alto * ancho * 3 / 1e9
        print(f"{nombre:<10}{a:<6}{p['macs'] / 1e6:>8.1f}{p['flash_kb']:>9.0f}"
              f"{arena:>9.0f}{lugar:>7}{lat:>10.0f}{cache_gb:>9.1f}")

### Presupuesto de memoria en el nodo

Ese veredicto se traduce en un reparto concreto, y condiciona el firmware:

| Qué | Dónde | Cómo |
|---|---|---|
| Pesos (~206 kB int8) | **flash** | array `const` + `alignas(16)` (notebook 5): va a `.rodata`, no ocupa RAM |
| Arena del intérprete (~585 kB estimada) | **PSRAM** | `MALLOC_CAP_SPIRAM`: no entra en los ~250 kB de SRAM libre; en PSRAM ESP-NN pierde algo de velocidad, irrelevante acá (una foto cada varios minutos) |
| Framebuffer del OV2640 | **PSRAM** | QVGA RGB565 = 153 600 B (150 kB) por buffer, y el sensor pide varios |

Los ~250 kB de SRAM libre son una **estimación**, no una medición: se confirma en placa con
`heap_caps_get_largest_free_block(MALLOC_CAP_INTERNAL)` y con
`interpreter->arena_used_bytes()` después de `AllocateTensors()`.

## 5. Entrenamiento

El entrenamiento va en **dos etapas**, porque el backbone viene preentrenado en ImageNet
y la cabeza arranca al azar:

1. **Cabeza sola** (`backbone.trainable = False`, lr = 1e-3). Los gradientes de una capa
   de salida aleatoria son enormes; si el backbone estuviera suelto desde el primer lote,
   esos gradientes borrarían justamente las características que se fueron a buscar a
   ImageNet. Además es barata: no se acumulan gradientes de los 14 bloques.
2. **Ajuste fino completo** (`trainable = True`, lr = 1e-4). Con la cabeza ya razonable,
   se suelta todo con la décima parte de la tasa de aprendizaje. Hay que **recompilar**:
   si no, el cambio de `trainable` no entra en el grafo de entrenamiento.

Cada etapa corre con los mismos dos *callbacks*:

- **`EarlyStopping`** corta cuando la pérdida de validación deja de mejorar durante 8
  épocas y restaura los mejores pesos, así que fijar un número alto de épocas no cuesta
  nada.
- **`ReduceLROnPlateau`** reduce a la mitad la tasa de aprendizaje ante un estancamiento,
  para afinar el ajuste en las últimas épocas.

La pérdida es **`sparse_categorical_crossentropy`**: la etiqueta es el índice entero de
la categoría y cada imagen pertenece a una sola clase. Y ahora **`class_weight` SÍ
sirve**: con una clase entera por imagen se usan pesos balanceados n/(3·n_c), para que la
red no aprenda a ignorar la clase escasa (`incendio`, ~27 % del banco). La métrica es
`SparseCategoricalAccuracy`: compara el argmax con el índice, que es la regla de
decisión que se va a usar.

In [ ]:
# Topes, no objetivos: EarlyStopping decide. En la corrida anterior la etapa 2 terminó
# en la época 45 de 45 sin que EarlyStopping llegara a dispararse —el modelo seguía
# mejorando y se quedó sin presupuesto—, así que el tope estaba atando el resultado.
EPOCAS_CABEZA = 15      # etapa 1: sólo la cabeza, backbone congelado
EPOCAS_FINO = 120       # etapa 2: ajuste fino de todo con lr/10

# class_weight SÍ vale con una clase por imagen: pesos balanceados n/(3·n_c) para que
# la red no ignore la clase escasa.
conteos_train = np.bincount(CAT[idx_train], minlength=len(CATEGORIAS))
PESOS_CLASE = {c: len(idx_train) / (len(CATEGORIAS) * n) for c, n in enumerate(conteos_train)}
print({c: f"{conteos_train[i] / len(idx_train):.1%} en train, peso {PESOS_CLASE[i]:.2f}"
       for i, c in enumerate(CATEGORIAS)})


def entrenar(epocas, etiqueta):
    print(f"\n--- {etiqueta} ---")
    return modelo_train.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epocas,
        class_weight=PESOS_CLASE,
        # Una línea por época: la barra por lote mete ~17 000 actualizaciones en la
        # salida del notebook (varios MB versionados en git) y no agrega información.
        verbose=2,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1
            ),
        ],
    ).history


# Etapa 1: sólo la cabeza. Barata (el backbone no acumula gradientes) y deja los
# pesos de la salida en un punto razonable antes de tocar ImageNet.
h1 = entrenar(EPOCAS_CABEZA, f"etapa 1: cabeza sola, lr=1e-3, hasta {EPOCAS_CABEZA} épocas")

# Etapa 2: ajuste fino de todo con lr/10. Recompilar es obligatorio para que el
# cambio de `trainable` entre en el grafo de entrenamiento.
modelo_base.get_layer("mobilenet").trainable = True
modelo_train.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-4,
        use_ema=True,
        ema_momentum=0.99,
        ema_overwrite_frequency=len(train_ds),
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
h2 = entrenar(EPOCAS_FINO, f"etapa 2: ajuste fino completo, lr=1e-4, hasta {EPOCAS_FINO} épocas")

historia = {k: h1[k] + h2[k] for k in h1}
CORTE_ETAPAS = len(h1["loss"])      # dónde empezó el ajuste fino, para el gráfico
EPOCAS_REALES = len(historia["loss"])
print(f"\n{EPOCAS_REALES} épocas en total ({CORTE_ETAPAS} de cabeza + "
      f"{EPOCAS_REALES - CORTE_ETAPAS} de ajuste fino)")

In [ ]:
h = historia
epocas = range(1, len(h["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (metrica, titulo) in zip(axes, [("loss", "Pérdida"), ("accuracy", "Exactitud")]):
    ax.plot(epocas, h[metrica], label="entrenamiento")
    ax.plot(epocas, h["val_" + metrica], label="validación")
    ax.set_xlabel("época")
    ax.set_title(titulo)
    ax.axvline(CORTE_ETAPAS + 0.5, color="gray", ls="--", lw=1, label="ajuste fino")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

La distancia entre ambas curvas indica el grado de sobreajuste. Si la de validación se
despega hacia arriba mientras la de entrenamiento sigue bajando, conviene subir el
`Dropout` o intensificar el aumento de datos.

## 6. Evaluación sobre el conjunto de prueba

El conjunto de prueba no participó del entrenamiento ni de la selección de pesos, así
que es la estimación honesta del desempeño. Se evalúa **`modelo_base`**: el aumento sólo
existe en entrenamiento y no tiene nada que hacer acá.

`test_ds` se armó sin mezclar, así que las predicciones salen en el mismo orden que
`idx_test` y las etiquetas verdaderas se leen directamente de `CAT[idx_test]`.


In [ ]:
y_prob = modelo_base.predict(test_ds, verbose=0)    # (N, 3): P(normal), P(humo), P(incendio)
y_pred = y_prob.argmax(axis=1)
y_true = CAT[idx_test]
COMBO_TEST = combo(ETIQUETAS[idx_test])     # para desglosar `incendio` en fuego / fuego+humo
print(f"{len(y_true)} imágenes evaluadas")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix


def informe(y_t, y_p, combos):
    '''Métricas de 3 clases en texto, más lo que el reporte no muestra.

    Tres líneas propias: % de `incendio` que sale como `normal` (alerta perdida, el
    error caro), % de `normal` que sale como humo/incendio (falsa alarma), y recall de
    `incendio` separado en fuego-sin-humo (combo 2) y fuego+humo (combo 3): dice si la
    red detecta la llama suelta o sólo cuando viene con humo.
    '''
    print(classification_report(y_t, y_p, labels=range(len(CATEGORIAS)),
                                target_names=CATEGORIAS, digits=3, zero_division=0))
    inc = y_t == 2
    nor = y_t == 0
    print(f"incendio → normal (alerta perdida): {np.mean(y_p[inc] == 0):.1%}")
    print(f"normal → humo/incendio (falsa alarma): {np.mean(y_p[nor] != 0):.1%}")
    for nombre, c in [("fuego-sin-humo", 2), ("fuego+humo", 3)]:
        m = inc & (combos == c)
        print(f"recall incendio {nombre}: "
              f"{np.mean(y_p[m] == 2):.1%} sobre {m.sum()} imágenes" if m.sum()
              else f"recall incendio {nombre}: sin imágenes")


informe(y_true, y_pred, COMBO_TEST)

In [ ]:
# Matriz 3×3 sobre categorías: deja ver, por ejemplo, cuántos incendios la red
# vio sólo como humo (alerta igual) y cuántos como normal (alerta perdida).
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (norma, formato, titulo) in zip(
    axes,
    [(None, "d", "Matriz de confusión (absoluta)"),
     ("true", ".2f", "Normalizada por fila")],
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred, labels=range(len(CATEGORIAS)),
                         normalize=norma),
        display_labels=CATEGORIAS,
    ).plot(ax=ax, cmap="Blues", colorbar=False, values_format=formato)
    ax.set_title(titulo)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 6bis. Cuánto del conjunto de prueba es un gemelo del entrenamiento

D-Fire sale en buena parte de cuadros de video, y eso deja imágenes casi idénticas
repartidas por todo el banco. Un *dhash* de 64 bits (gradiente horizontal sobre una
miniatura de 9×8 en gris) colapsa a un mismo entero las imágenes visualmente
equivalentes; con 21 527 imágenes la probabilidad de una colisión por azar es del orden
de 10⁻¹¹, así que dos hashes iguales son un gemelo real, no un accidente.

Acertar sobre un gemelo no prueba que la red generalice: prueba que memorizó. La cifra
de abajo dice qué parte del conjunto de prueba está en esa situación, y el informe
siguiente es la métrica honesta: la misma red medida sólo sobre las imágenes de prueba
que **no** tienen gemelo en entrenamiento.


In [ ]:
from dfire import sin_gemelo

# `sin_gemelo` compara el dhash de cada imagen de prueba contra todos los de
# entrenamiento (la función está en `modulos/dfire.py`).
LIMPIO = sin_gemelo(ARCHIVOS[idx_train], ARCHIVOS[idx_test])

print(f"prueba: {len(LIMPIO)} imágenes, {np.sum(~LIMPIO)} con gemelo en entrenamiento "
      f"({100 * np.mean(~LIMPIO):.1f} %), {np.sum(LIMPIO)} limpias\n")
print("=== float32, sólo sobre las imágenes de prueba SIN gemelo ===")
informe(y_true[LIMPIO], y_pred[LIMPIO], COMBO_TEST[LIMPIO])

El error más costoso es un `incendio` que la red ve como **`normal`**: el sistema no
dispara la alerta. Que un incendio salga como `humo` casi no cuesta —la alerta sale igual,
y el humo suele anteceder a la llama—.

Por eso la métrica que decide no es la exactitud sino qué pasa con la alerta, y el umbral
de decisión se fija sobre el modelo **int8** (notebook 5), que es el que corre en el nodo.

In [ ]:
# Peores errores: imágenes mal clasificadas donde la red más se alejó de la
# verdad (1 - P(clase real)).
distancia = 1 - y_prob[np.arange(len(y_true)), y_true]
errores = np.flatnonzero(y_pred != y_true)
peores = errores[np.argsort(-distancia[errores])][:8]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.ravel(), peores):
    ax.imshow(keras.utils.load_img(ARCHIVOS[idx_test][idx]))
    probs = "   ".join(f"P({c}) {y_prob[idx, i]:.0%}" for i, c in enumerate(CATEGORIAS))
    ax.set_title(f"real: {CATEGORIAS[y_true[idx]]}\n{probs}", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(peores):]:
    ax.axis("off")
plt.suptitle("Errores con mayor confianza")
plt.tight_layout()
plt.show()

### Fallos de incendio según el tamaño de la llama

La categoría `incendio` vale si la imagen trae **al menos una caja** de fuego, sin
importar su tamaño. El tamaño se mide **relativo** (fracción del lado, `sqrt(w·h)`
normalizado) para que el mismo corte compare corridas a distinta resolución; la columna
de px lo traduce a esta resolución (< 5 % del lado ≈ 5 px a 96×96: ahí no queda llama
visible que aprender). Si el recall se cae sólo en las cajas chicas, esos fallos son
ruido de etiqueta del banco y no un defecto de la red; la columna de alerta dice si el
sistema los rescata igual por el humo.

In [ ]:
from dfire import cajas_imagen


def lado_fuego_rel(ruta_img):
    '''Lado equivalente de la caja de fuego más grande, como fracción del lado.

    Las coordenadas YOLO van normalizadas a [0, 1], así que `sqrt(w·h)` no depende de
    la resolución: el mismo corte compara corridas a distinto tamaño. 0 si no hay caja.
    '''
    txt = Path(ruta_img).parent.parent / "labels" / (Path(ruta_img).stem + ".txt")
    cajas = np.asarray(cajas_imagen(txt), dtype=float).reshape(-1, 5)
    fuego = cajas[cajas[:, 0] == 0]     # canal 0 = fuego
    return float(np.sqrt(np.max((fuego[:, 3] - fuego[:, 1])
                                * (fuego[:, 4] - fuego[:, 2])))) if len(fuego) else 0.0


con_incendio = np.flatnonzero(y_true == 2)
lado = np.array([lado_fuego_rel(ARCHIVOS[idx_test][i]) for i in con_incendio])
vio_incendio = y_pred[con_incendio] == 2
alerta = y_pred[con_incendio] != 0      # humo o incendio: la alerta sale igual

CORTES = [0, 0.03, 0.05, 0.10, 0.20, 1]
escala = np.sqrt(IMG_SIZE[0] * IMG_SIZE[1])     # px equivalentes a esta resolución

print(f"{len(con_incendio)} imágenes de prueba con incendio\n")
print(f"{'lado relativo':<14}{'~px acá':<12}{'imágenes':>9}"
      f"{'recall incendio':>16}{'con alerta':>12}")
for lo, hi in zip(CORTES[:-1], CORTES[1:]):
    m = (lado >= lo) & (lado < hi)
    rango = f"{lo:.0%}–{hi:.0%}"
    px = f"~{lo * escala:.0f}–{hi * escala:.0f} px"
    n = m.sum()
    rec = f"{vio_incendio[m].mean():.1%}" if n else "—"
    ale = f"{alerta[m].mean():.1%}" if n else "—"
    print(f"{rango:<14}{px:<12}{n:>9}{rec:>16}{ale:>12}")

## 7. Interpretabilidad: Grad-CAM

Una exactitud alta no garantiza que la red mire lo correcto: podría estar aprovechando
artefactos de adquisición correlacionados con la clase. **Grad-CAM** produce un mapa de
calor de las regiones que más pesaron en la decisión, en tres pasos:

1. Calcular el gradiente de la probabilidad de **una clase** respecto de los mapas de características
   de la última capa convolucional.
2. Promediar ese gradiente sobre el espacio: da un peso por canal, que mide cuánto
   contribuye ese canal a la clase.
3. Sumar los mapas ponderados por esos pesos y quedarse con la parte positiva (ReLU),
   que es la evidencia a favor de la clase.

Con softmax sí hay una clase predicha, pero para validar que la red distingue las dos
señales se pide un mapa **por clase**: abajo, `humo` e `incendio` sobre la misma imagen.

In [ ]:
CAPA_GRAD_CAM = "conv_pw_11_relu"   # grilla de 15×20 a QVGA; la última (conv_pw_13) da 8×10 y no se lee


def grad_cam(modelo, imagen, clase):
    '''Mapa de calor Grad-CAM de UNA clase (índice en CATEGORIAS) para una imagen
    (H, W, 3) cruda, en 0-255.

    La clase va explícita: con el argmax solo se vería la ganadora y no se sabría si
    la red distingue el humo de la llama en la misma escena.

    MobileNet va anidada dentro de `modelo_base`, así que su grafo interno no cuelga de
    la entrada del modelo: el submodelo se arma sobre el backbone y se le vuelve a colgar
    la cabeza reutilizando las MISMAS capas (`dropout` y `salida`), o sea los pesos
    entrenados. `GlobalAveragePooling2D` no tiene pesos, se puede crear de nuevo.

    La cabeza cuelga de `backbone.output`, NO de los mapas elegidos: el backbone corre
    entero y la `Dense` de salida recibe los 256 canales que espera. Los mapas
    intermedios salen como una segunda salida del mismo submodelo. Hacerlo al revés
    —colgar la cabeza de los mapas— sólo funciona si la capa elegida es la última, y
    revienta con cualquier otra (`conv_pw_11_relu` da 128 canales, no 256).
    '''
    backbone = modelo.get_layer("mobilenet")
    mapas = backbone.get_layer(CAPA_GRAD_CAM).output
    x = layers.GlobalAveragePooling2D(name="gap_cam")(backbone.output)
    x = modelo.get_layer("dropout")(x, training=False)
    puntajes = modelo.get_layer("salida")(x)
    submodelo = keras.Model(backbone.input, [mapas, puntajes])

    # Misma escala que la capa `Rescaling` del modelo: (pixel / 127.5) - 1.
    x = tf.expand_dims(imagen / 127.5 - 1.0, 0)

    with tf.GradientTape() as cinta:
        mapas_act, predicciones = submodelo(x, training=False)
        score = predicciones[:, clase]

    # d(score)/d(mapas): cuánto cambia la probabilidad de la clase por unidad de activación
    gradientes = cinta.gradient(score, mapas_act)
    pesos = tf.reduce_mean(gradientes, axis=(0, 1, 2))          # promedio espacial
    mapa = tf.reduce_sum(mapas_act[0] * pesos, axis=-1)         # combinación ponderada
    mapa = tf.nn.relu(mapa)                                     # sólo evidencia positiva

    mapa = mapa / (tf.reduce_max(mapa) + keras.backend.epsilon())
    return mapa.numpy()

In [ ]:
from dfire import leer_imagen

# Una imagen de prueba por categoría real y un mapa por clase (`humo` e `incendio`):
# en un incendio con humo tienen que encenderse los dos. Cada mapa se normaliza a su
# máximo, así que siempre hay una zona "caliente": si la P del título es baja (p. ej.
# el mapa de incendio en una `normal`), el mapa no significa nada.
FILAS_CAM = [1, 2]      # humo, incendio
muestras = [np.flatnonzero(y_true == c)[0] for c in range(len(CATEGORIAS))]

fig, axes = plt.subplots(len(FILAS_CAM), len(muestras),
                         figsize=(3.2 * len(muestras), 4.5))
for col, i in enumerate(muestras):
    # Mismo preprocesamiento que test_ds; y_prob está en el orden de idx_test.
    imagen = tf.cast(leer_imagen(ARCHIVOS[idx_test][i], IMG_SIZE), tf.float32)
    correcto = "OK" if y_pred[i] == y_true[i] else "ERROR"
    for fila, clase in enumerate(FILAS_CAM):
        mapa = grad_cam(modelo_base, imagen, clase)
        # El mapa es de 15×20 (resolución de conv_pw_11 a QVGA); lo estiramos a la imagen.
        mapa_grande = tf.image.resize(mapa[..., None], IMG_SIZE).numpy().squeeze()
        ax = axes[fila, col]
        ax.imshow(imagen.numpy().astype("uint8"))
        ax.imshow(mapa_grande, cmap="jet", alpha=0.4)
        cabecera = f"real: {CATEGORIAS[y_true[i]]}  [{correcto}]\n" if fila == 0 else ""
        ax.set_title(f"{cabecera}mapa de {CATEGORIAS[clase]}: P {y_prob[i, clase]:.0%}",
                     fontsize=9)
        ax.axis("off")

plt.suptitle("Grad-CAM por clase: una imagen de prueba por categoría real")
plt.tight_layout()
plt.show()

Si las zonas calientes coinciden con llamas o penachos de humo, la red aprendió
las características relevantes. Si se concentran en el cielo, los bordes o regiones
fijas del cuadro, está explotando atajos del dataset y el resultado no generalizaría
a las cámaras del nodo LoRa.

## 8. Entregable: el modelo entrenado

La salida de este notebook es el **`.keras` en float32**: el *checkpoint* de
`modelo_base`, el modelo sin las capas de aumento. No va al nodo tal cual:
`5_cuantizacion.ipynb` lo toma (el `cnn_fuego_3clases_*` más reciente), lo cuantiza a
int8, lo mide como lo va a ver la cámara, fija el umbral de decisión y genera los
archivos para el firmware.

| Archivo | Qué es |
|---|---|
| `salidas/modelos/cnn_fuego_3clases_epochs_*_image_size_320x240.keras` | modelo entrenado (ancho×alto); el notebook 5 toma el más reciente |

Además cada corrida agrega una fila a `salidas/barrido_resolucion.csv` (se crea con
encabezado si no existe): fecha, archivo, alto, ancho, alpha, costo en el nodo (MMAC,
arena, memoria, latencia) y métricas limpias (exactitud, F1 macro, recall de incendio,
incendio→normal, falsa alarma, recall con caja chica). Es la tabla costo/resolución
entre corridas: correr una vez con `(96, 96)` da la línea base (§9).

In [ ]:
# `EPOCAS_REALES`, no el tope: EarlyStopping corta antes y el nombre tiene que decir
# cuántas épocas se entrenaron de verdad. Ancho×alto en el nombre, como los framesizes.
import csv
import datetime

from sklearn.metrics import f1_score

ruta_modelos = SALIDAS / "modelos"
ruta_modelos.mkdir(parents=True, exist_ok=True)
ruta_keras = (ruta_modelos
              / f"cnn_fuego_3clases_epochs_{EPOCAS_REALES}"
              f"_image_size_{IMG_SIZE[1]}x{IMG_SIZE[0]}.keras")
modelo_base.save(ruta_keras)
print(f"{ruta_keras}  ({ruta_keras.stat().st_size / 1024:.0f} kB)")

# Una fila por corrida en el barrido costo/resolución: métricas sobre la prueba limpia
# y recall con caja chica (< 5 % del lado) sobre la prueba completa.
yt_limpio, yp_limpio = y_true[LIMPIO], y_pred[LIMPIO]
m_inc_limpio = yt_limpio == 2
m_nor_limpio = yt_limpio == 0
fila = {
    "fecha": datetime.date.today().isoformat(),
    "archivo": ruta_keras.name,
    "alto": IMG_SIZE[0],
    "ancho": IMG_SIZE[1],
    "alpha": ALPHA,
    "mmac": round(perfil_esp32["macs"] / 1e6, 1),
    "arena_kb": round(arena_kb),
    "memoria": donde,
    "latencia_ms": round(latencia_s3),
    "epocas": EPOCAS_REALES,
    "exactitud_limpia": round(float(np.mean(yp_limpio == yt_limpio)), 4),
    "f1_macro_limpia": round(float(f1_score(yt_limpio, yp_limpio, labels=[0, 1, 2],
                                            average="macro", zero_division=0)), 4),
    "recall_incendio_limpia": round(float(np.mean(yp_limpio[m_inc_limpio] == 2)), 4),
    "incendio_a_normal_limpia": round(float(np.mean(yp_limpio[m_inc_limpio] == 0)), 4),
    "falsa_alarma_limpia": round(float(np.mean(yp_limpio[m_nor_limpio] != 0)), 4),
    "recall_incendio_caja_chica": (round(float(vio_incendio[lado < 0.05].mean()), 4)
                                   if np.sum(lado < 0.05) else ""),
}
ruta_barrido = SALIDAS / "barrido_resolucion.csv"
nueva = not ruta_barrido.exists()
with open(ruta_barrido, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(fila))
    if nueva:
        w.writeheader()
    w.writerow(fila)
print(f"{ruta_barrido}: fila agregada ({fila['archivo']})")

## 9. Conclusiones y posibles extensiones

Lo construido: un clasificador de 3 clases —P(`normal`), P(`humo`), P(`incendio`),
softmax— entrenado sobre D-Fire a QVGA 320×240, con el pipeline completo de datos,
entrenamiento regularizado con `class_weight`, evaluación sobre un conjunto reservado
(completa y sin gemelos) e inspección de lo que la red aprendió (Grad-CAM). La
cuantización a **TFLite Micro int8** para el ESP32-S3, y todo lo que mide al modelo tal
como corre en el nodo, están en `5_cuantizacion.ipynb`. Es el eslabón de visión del
Proyecto Final (red LoRa de detección temprana de incendios): queda validado que la
señal visual alcanza para discriminar *y* que el modelo entra en el nodo (arena en
PSRAM, de sobra en 8 MB).

Cómo comparar resoluciones: cada corrida deja su fila en `salidas/barrido_resolucion.csv`;
correr una vez con `IMG_SIZE = (96, 96)` da la línea base contra la que se mide QVGA
(misma red, misma receta, sólo cambia `tam`).

Dos resultados condicionan cómo leer los números:

- **Tres clases porque el humo acompaña al fuego.** El 80 % de las escenas con llama
  también tiene humo. Separar `fuego` de `fuego+humo` partía en dos la clase más escasa;
  con `incendio` = llama con o sin humo la clase llega al 27 % del banco, y que un
  incendio salga como `humo` igual dispara la alerta.
- **El banco se contamina a sí mismo.** El 22,9 % de las imágenes de prueba tiene un
  gemelo perceptual en entrenamiento (§6bis), aun respetando la partición oficial. Las
  métricas se informan con y sin esos gemelos; la que vale es la segunda.

Siguientes palancas (no implementadas acá):

- **α=0.5 a QVGA** (230 MMAC, 796 kB de pesos, ~1,2 MB de arena, ~2,2 s) si QVGA con
  α=0.25 se estanca: sigue entrando en los 8 MB de PSRAM y el tiempo no importa.
- **Incendios lejanos: mosaicos QVGA.** Para ver más lejos, recorrer la foto grande del
  OV2640 (UXGA/SVGA) en mosaicos QVGA con este mismo modelo: ~25 mosaicos de ~0,6 s son
  unos 15 s por barrido, aceptable si hay una foto cada varios minutos.